# 07 Inference variants before a submission

> **Earlier version, kept for its results.** This notebook ran on Kaggle with the code of commit `9b394a7` (package `src/vod`, later replaced by `src/core`). Validation here is the official val split, not the folds. To rerun it, clone that commit instead of `main`.

In [1]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

Cloning into '/tmp/repo'...


In [2]:
!pip install -q ultralytics faster-coco-eval

In [3]:
import ast
import shutil

import pandas as pd
import torch

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.evaluate import coco_evaluate, per_clip_map
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import comparison_tables, completed_runs, load_run_model
from vod.variants import (choose_variant, collect_view, compare_variants, merge_views, torchvision_detector,
                          yolo_detector)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
ROOTS = (WORK, Path("/kaggle/input"))

## Models and variants

In [4]:
VAL_STEP = 5
LOW_RES = 960  # test has 680 and 960 px clips, val starts at 1344
BATCH = 4
REF = "full"
VARIANTS = {
    REF: dict(views=["full"]),
    "flip_nms50": dict(views=["full", "flip"], iou=0.5),
    "flip_nms65": dict(views=["full", "flip"], iou=0.65),
    "tiles_nms50": dict(views=["full", "tiles"], iou=0.5),
    "tiles_nms65": dict(views=["full", "tiles"], iou=0.65),
    "flip_tiles_nms50": dict(views=["full", "flip", "tiles"], iou=0.5),
    "flip_tiles_nms65": dict(views=["full", "flip", "tiles"], iou=0.65),
}
VIEWS = sorted({n for v in VARIANTS.values() for n in v["views"]})

In [5]:
def has_weights(pattern):
    return any(root.exists() and any(root.rglob(pattern)) for root in ROOTS)


def make_detector(name):
    if name.startswith(("yolo", "rtdetr")):
        from vod.yolo import load_ultralytics
        pattern = "baseline/weights/best.pt" if name == "yolo_baseline" else f"{name}/weights/last.pt"
        weights = next(p for root in ROOTS if root.exists() for p in root.rglob(pattern))
        return yolo_detector(load_ultralytics(weights))
    cfg = runs[name]["config"]
    return torchvision_detector(load_run_model(name, cfg, ROOTS), cfg)


runs = completed_runs(ROOTS)
candidates = ["yolo_baseline"] if has_weights("baseline/weights/best.pt") else []
if runs:
    main = comparison_tables(runs, CLASSES)[0]
    for prefix, pattern in [("yolo_", "{}/weights/last.pt"), ("rtdetr", "{}/weights/last.pt"),
                            ("frcnn", "{}/last.pt"), ("retina", "{}/last.pt"), ("fcos", "{}/last.pt")]:
        family = [n for n in main[main.index.str.startswith(prefix)].sort_values("mAP", ascending=False).index
                  if has_weights(pattern.format(n))]
        candidates += family[:1]
print("candidates:", candidates)

candidates: ['yolo_m', 'rtdetr_lr3e4']


## Comparison on val: native resolution and shrunk to the low-res test clips

In [6]:
val_frames, _ = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
paths = val_frames.iloc[::VAL_STEP]["path"].tolist()
print(len(paths), "val frames")

results = {}
for name in candidates:
    detect = make_detector(name)
    views_native = {v: collect_view(detect, paths, v, batch=BATCH) for v in VIEWS}
    views_lowres = {v: collect_view(detect, paths, v, max_side=LOW_RES, batch=BATCH) for v in VIEWS}
    native = compare_variants(views_native, VARIANTS, VAL_GT, paths, REF)
    lowres = compare_variants(views_lowres, VARIANTS, VAL_GT, paths, REF, clips=False)
    chosen = choose_variant(native, lowres, REF)
    pick = VARIANTS[chosen]
    results[name] = dict(native=native, lowres=lowres, chosen=chosen,
                         dets_native=merge_views([views_native[v] for v in pick["views"]], pick.get("iou")),
                         dets_lowres=merge_views([views_lowres[v] for v in pick["views"]], pick.get("iou")))
    native.to_csv(WORK / f"variants_{name}_native.csv")
    lowres.to_csv(WORK / f"variants_{name}_lowres.csv")
    print(name, "| chosen:", chosen)
    display(native)
    display(lowres)
    del detect
    torch.cuda.empty_cache()

570 val frames
yolo_m | chosen: flip_tiles_nms65


,mAP,AP50,AP75,AP_small,AR500,ms_per_frame,clips_better
full,0.2213,0.4332,0.1954,0.1520,0.3462,86.2,0
flip_nms50,0.2274,0.4487,0.2003,0.1574,0.3519,212.2,7
flip_nms65,0.2294,0.4501,0.2024,0.1583,0.3645,212.2,7
tiles_nms50,0.2503,0.4839,0.2277,0.1324,0.3931,365.6,6
tiles_nms65,0.2524,0.4838,0.2305,0.1337,0.4028,365.6,7
flip_tiles_nms50,0.2511,0.4854,0.2273,0.1435,0.3975,491.6,7
flip_tiles_nms65,0.2534,0.4850,0.2303,0.1450,0.4100,491.6,7


,mAP,AP50,AP75,AP_small,AR500,ms_per_frame
full,0.2256,0.4310,0.2081,0.1479,0.3435,99.7
flip_nms50,0.2304,0.4484,0.2069,0.1502,0.3445,201.3
flip_nms65,0.2326,0.4483,0.2094,0.1514,0.3575,201.3
tiles_nms50,0.2491,0.4678,0.2366,0.1272,0.3857,384.1
tiles_nms65,0.2510,0.4668,0.2392,0.1282,0.3964,384.1
flip_tiles_nms50,0.2529,0.4744,0.2390,0.1297,0.3930,485.7
flip_tiles_nms65,0.2548,0.4728,0.2422,0.1307,0.4038,485.7


rtdetr_lr3e4 | chosen: flip_tiles_nms65


,mAP,AP50,AP75,AP_small,AR500,ms_per_frame,clips_better
full,0.2641,0.4978,0.2452,0.1178,0.4074,176.1,0
flip_nms50,0.2791,0.5244,0.2593,0.1292,0.4135,382.0,7
flip_nms65,0.2803,0.5255,0.2607,0.1295,0.4202,382.0,7
tiles_nms50,0.2818,0.5352,0.2540,0.1718,0.4384,805.2,7
tiles_nms65,0.2830,0.5358,0.2555,0.1723,0.4480,805.2,7
flip_tiles_nms50,0.2855,0.5396,0.2593,0.1762,0.4403,1011.1,7
flip_tiles_nms65,0.2870,0.5404,0.2610,0.1769,0.4514,1011.1,7


,mAP,AP50,AP75,AP_small,AR500,ms_per_frame
full,0.2545,0.4804,0.2360,0.1105,0.3925,185.3
flip_nms50,0.2655,0.5016,0.2455,0.1181,0.3978,374.4
flip_nms65,0.2663,0.5018,0.2465,0.1190,0.4040,374.4
tiles_nms50,0.2602,0.4963,0.2362,0.1321,0.4174,818.6
tiles_nms65,0.2613,0.4967,0.2370,0.1324,0.4276,818.6
flip_tiles_nms50,0.2637,0.5024,0.2403,0.1365,0.4217,1007.7
flip_tiles_nms65,0.2649,0.5027,0.2417,0.1371,0.4324,1007.7


In [7]:
summary = pd.DataFrame({
    n: {"chosen": r["chosen"],
        "mAP_ref": r["native"].loc[REF, "mAP"], "mAP": r["native"].loc[r["chosen"], "mAP"],
        "mAP_lowres_ref": r["lowres"].loc[REF, "mAP"], "mAP_lowres": r["lowres"].loc[r["chosen"], "mAP"],
        "clips_better": r["native"].loc[r["chosen"], "clips_better"],
        "ms_per_frame": r["native"].loc[r["chosen"], "ms_per_frame"]}
    for n, r in results.items()}).T
summary["score"] = (summary["mAP"] + summary["mAP_lowres"]) / 2
summary = summary.sort_values("score", ascending=False)
summary

,chosen,mAP_ref,mAP,mAP_lowres_ref,mAP_lowres,clips_better,ms_per_frame,score
rtdetr_lr3e4,flip_tiles_nms65,0.2641,0.287,0.2545,0.2649,7,1011.1,0.27595
yolo_m,flip_tiles_nms65,0.2213,0.2534,0.2256,0.2548,7,491.6,0.2541


## Ensembles of the best models
Must beat the best single model at both resolutions and in most clips.

In [8]:
ENS_IOU = 0.5
best = summary.index[0]
best_clips = pd.Series(per_clip_map(VAL_GT, results[best]["dets_native"], paths))
ensembles = {f"ens_top{k}": list(summary.index[:k]) for k in range(2, len(summary) + 1)}
rows = {}
for ens, members in ensembles.items():
    native = merge_views([results[n]["dets_native"] for n in members], ENS_IOU)
    lowres = merge_views([results[n]["dets_lowres"] for n in members], ENS_IOU)
    clips = pd.Series(per_clip_map(VAL_GT, native, paths))
    rows[ens] = {"mAP": coco_evaluate(VAL_GT, native, file_names=paths)["mAP"],
                 "mAP_lowres": coco_evaluate(VAL_GT, lowres, file_names=paths)["mAP"],
                 "clips_better": int((clips > best_clips).sum())}
ens_table = pd.DataFrame(rows).T
if len(ens_table):
    ens_table["score"] = (ens_table["mAP"] + ens_table["mAP_lowres"]) / 2
    display(ens_table.round(4))
pd.concat([summary, ens_table]).to_csv(WORK / "variants_summary.csv")

,mAP,mAP_lowres,clips_better,score
ens_top2,0.2891,0.2763,5.0,0.2827


## Test predictions of the winner

In [9]:
WINNER = [best]
if len(ens_table):
    ok = ens_table[(ens_table["mAP"] > summary.loc[best, "mAP"])
                   & (ens_table["mAP_lowres"] > summary.loc[best, "mAP_lowres"])
                   & (ens_table["clips_better"] > len(best_clips) / 2)]
    if len(ok):
        WINNER = ensembles[ok["score"].idxmax()]
print("winner:", {n: results[n]["chosen"] for n in WINNER})

test = pd.read_csv(DATA / "test_images.csv")
test_paths = [str(DATA / f) for f in test["file_name"]]
test_views = {}


def submit(members, variant=None):
    parts = []
    for name in members:
        pick = VARIANTS[variant or results[name]["chosen"]]
        missing = [v for v in pick["views"] if (name, v) not in test_views]
        if missing:
            detect = make_detector(name)
            for v in missing:
                test_views[name, v] = collect_view(detect, test_paths, v, batch=BATCH)
            del detect
            torch.cuda.empty_cache()
        parts.append(merge_views([test_views[name, v] for v in pick["views"]], pick.get("iou")))
    dets = merge_views(parts, ENS_IOU if len(parts) > 1 else None)
    dets["image_id"] = dets.pop("file_name").map(dict(zip(test_paths, test["image_id"])))
    tag = "+".join(members) + (f"_{variant}" if variant else "")
    return write_submission(dets, DATA / "test_images.csv", WORK / f"submission_{tag}.csv")


sub = submit(WINNER)
# val ranks the models differently from the leaderboard, so these are written whatever val says
RTDETR = [n for n in candidates if n.startswith("rtdetr")][:1]
EXTRA = [(["yolo_baseline", "frcnn_default", "fcos_default"], None), (candidates, None),
         (RTDETR, "flip_nms65"), (RTDETR, "flip_tiles_nms65")]
for members, variant in EXTRA:
    if members and set(members) <= set(candidates) and (variant or set(members) != set(WINNER)):
        submit(members, variant)
sub.head()

winner: {'rtdetr_lr3e4': 'flip_tiles_nms65', 'yolo_m': 'flip_tiles_nms65'}


,image_id,PredictionString
0,clip_001_0000001,0 0.8503 782.1 887.7 61.9 163.3 0 0.8224 631.2...
1,clip_001_0000002,0 0.8471 780.1 891.4 62.4 168.2 0 0.8251 626.2...
2,clip_001_0000003,0 0.8439 775.0 898.0 64.2 163.5 0 0.8301 616.4...
3,clip_001_0000004,0 0.8063 762.9 902.9 62.9 175.6 0 0.7963 1669....
4,clip_001_0000005,0 0.8344 560.2 966.8 70.2 113.8 0 0.8294 730.0...


## Final model: retrained on train + val
The best RT-DETR, which is also the best single model on the leaderboard. No val number exists for it.

In [10]:
RETRAIN = RTDETR[0] if RTDETR else None


def parse(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return value


if RETRAIN:
    from vod.yolo import train_yolo, write_data_yaml, write_dataset
    cfg = {k: parse(v) for k, v in runs[RETRAIN]["config"].items()}
    tmp = Path("/tmp/final_data") if Path("/kaggle/working").exists() else WORK / "final_data"
    d = cfg.get("degrade_p", 0.0)
    train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
    val_frames, val_targets = load_split(DATA / "val", step=1)
    images = (write_dataset(train_frames, train_targets, tmp / "train", degrade_p=d)
              + write_dataset(val_frames.iloc[::PROTOCOL["train_step"]], val_targets, tmp / "valtrain", degrade_p=d))
    monitor = write_dataset(val_frames.iloc[::PROTOCOL["val_step_epoch"]], val_targets, tmp / "val")
    train_yolo(f"{RETRAIN}_final", cfg, write_data_yaml(tmp / "data.yaml", images, monitor), WORK)
    shutil.rmtree(tmp, ignore_errors=True)
    for variant in ["full", "flip_nms65"]:
        submit([f"{RETRAIN}_final"], variant)

10 epochs completed in 2.822 hours.
Validating /kaggle/working/runs/rtdetr_lr3e4_final/weights/best.pt...
Ultralytics 8.4.198 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 1.7it/s 21.6s
                   all        285      11112       0.84      0.829      0.848      0.508
                person        278       4874      0.808       0.75      0.804      0.388
               bicycle         97        593      0.822      0.804      0.829      0.455
                   car        238       3125      0.831      0.864      0.873      0.577
                   van        223        662      0.862      0.855      0.878      0.594
                 truck         99        137       0.89      0.876      0.902      0.572
              tricycle        166        506      0.881     